# Controlling Training with a Keras Callback

## Objective
Train a neural network on Fashion MNIST and use a custom callback to stop
training when training accuracy exceeds 60%.

In [22]:
import tensorflow as tf

## 1. Load and Prepare Fashion MNIST

Fashion MNIST contains 60,000 training images and 10,000 test images.
Each image is a 28 × 28 grayscale image with a label from 0 to 9.

The pixel values are divided by 255 to scale them to the range 0 to 1.

In [23]:
# Load dataset
fmnist=tf.keras.datasets.fashion_mnist

# Load train and test set
(train_image,train_label),(test_image,test_label)=fmnist.load_data()

# normalize pixel value
train_image=train_image/255.0
test_image=test_image/255.0

## 2. Create a Custom Callback

Keras calls `on_epoch_end()` after each training epoch. The callback checks
the training accuracy in `logs`. If accuracy is above 0.60, it sets
`self.model.stop_training` to `True` so training stops.

In [24]:
class MyCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        if logs["accuracy"] > 0.60:
            print(f"\nAccuracy exceeded 0.60; stopping after epoch {epoch + 1}.")
            self.model.stop_training = True



## 3. Build and Compile the Model

The model flattens each image, passes it through a 128-unit hidden layer,
and produces scores for 10 clothing classes.

It uses the Adam optimizer and sparse categorical cross-entropy loss.
Accuracy is tracked during training.

In [25]:
# define model 
model=tf.keras.models.Sequential([
    tf.keras.Input(shape=(28,28)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128,activation=tf.nn.relu),
    tf.keras.layers.Dense(10,activation=tf.nn.softmax)
])
model.compile(optimizer=tf.optimizers.Adam(),loss='sparse_categorical_crossentropy',metrics=['accuracy'])



## 4. Train with the Callback

The model can train for up to 10 epochs. After each epoch, the callback
checks whether training accuracy has exceeded 0.60. If so, it stops
training.

In [26]:
# train model
model.fit(
    train_image,
    train_label,
    epochs=10,
    callbacks=[MyCallback()],
    verbose=0
)


Accuracy exceeded 0.60; stopping after epoch 1.


## 5. Evaluate on the Test Images

Evaluate the model on the test images, which were not used for training.
This reports the model’s loss and accuracy on unseen examples.

In [27]:
test_loss, test_accuracy = model.evaluate(test_image, test_label, verbose=0)
print(f"Test accuracy: {test_accuracy:.2%}")

Test accuracy: 84.42%
